# UPMO: Uncertainty-Aware Multi-Objective Optimization for Smart Agriculture
### ACIT'2026 reproduction notebook

This notebook runs the full pipeline step by step:

1. Preprocessing (KNN imputation, lag and rolling features)
2. Control variable selection (Random Forest importance)
3. Gaussian Process surrogates, compared with baselines on a chronological split
4. Pareto-based multi-objective evolutionary search
5. Decision support (scenario profiles with GP uncertainty)

Run it from the `notebooks/` folder or open it in Colab after cloning the repo.

In [ ]:
# In Colab, uncomment:
# !git clone https://github.com/naimabouk/upmo-smart-agriculture.git
# %cd upmo-smart-agriculture
# !pip install -r requirements.txt

import os, sys, warnings
warnings.filterwarnings("ignore")
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import pandas as pd
from src import decision_support as ds
from src.preprocessing import prepare, split_features_targets
from src.feature_selection import rank_control_variables
from src.surrogate_models import scale_controls, temporal_split, compare_models, SurrogatePair
from src.optimization import run_optimization, pareto_metrics

## 1. Preprocessing

In [ ]:
DATA = "data/raw/synthetic_agritech_sensor_dataset.csv"
pd.read_csv(DATA).head()

In [ ]:
df_fe = prepare(DATA)
X, y_energy, y_yield = split_features_targets(df_fe)
print("Engineered dataset:", df_fe.shape)

## 2. Control variable selection

In [ ]:
importance = rank_control_variables(X, y_energy, y_yield)
control_vars = importance.head(10).index.tolist()
importance.head(10).sort_values().plot(kind="barh", figsize=(8, 5), title="Combined RF importance (top 10)")
plt.tight_layout(); plt.show()
X_ctrl = df_fe[control_vars]

In [ ]:
ds.plot_correlation_heatmap(df_fe, control_vars, show=True)

## 3. Gaussian Process surrogates (chronological 80/20 split)

In [ ]:
X_scaled, scaler = scale_controls(X_ctrl)
X_tr, X_te, yE_tr, yE_te, yY_tr, yY_te = temporal_split(X_scaled, y_energy, y_yield)
df_cmp, gp_energy, gp_yield = compare_models(X_tr, X_te, yE_tr, yE_te, yY_tr, yY_te)
df_cmp.sort_values(["Target", "MAE"])

In [ ]:
print("Energy kernel:", gp_energy.kernel_)
print("Yield kernel: ", gp_yield.kernel_)
ds.plot_actual_vs_pred(yE_te, gp_energy.predict(X_te), "Actual vs Predicted Energy (GP)", "", "Actual", "Predicted", show=True)
ds.plot_actual_vs_pred(yY_te, gp_yield.predict(X_te), "Actual vs Predicted Yield (GP)", "", "Actual", "Predicted", show=True)

## 4. Multi-objective optimization

Minimize energy and maximize yield. Set `BETA > 0` to penalize uncertain solutions
(fitness = mean ± beta·std). `BETA = 0` reproduces the paper's setting. This step takes a few minutes.

In [ ]:
BETA = 0.0
surrogates = SurrogatePair(gp_energy, gp_yield, scaler, control_vars)
df_pareto, history = run_optimization(surrogates, X_ctrl, pop_size=80, generations=50, beta=BETA, verbose=False)
pareto_metrics(df_pareto)

In [ ]:
ds.plot_pareto_front(df_pareto, show=True)
ds.plot_convergence(history, show=True)

## 5. Decision support

In [ ]:
scenarios = ds.profile_scenarios(df_pareto)
scenarios[["scenario", "pred_energy", "energy_std", "pred_yield", "yield_std"] + control_vars[:4]]

In [ ]:
ds.plot_scenario_uncertainty(df_pareto, show=True)

## Optional: SHAP explanation of the energy drivers
Requires `pip install shap`.

In [ ]:
try:
    import shap
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_ctrl, y_energy)
    shap.summary_plot(shap.TreeExplainer(rf).shap_values(X_ctrl), X_ctrl)
except ImportError:
    print("shap not installed: pip install shap")